In [57]:
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer, util
import torch

In [58]:
path_1 = "annotations/2-TVSUM-video_8-6310489.txt"
path_2 = "annotations/2-TVSUM-video_8-6328583.txt"

In [59]:
def parse_file(filepath):
    rows = []
    
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            #Remove whitespace or empty lines
            line = line.strip()
            if not line:
                continue
            
            #Split the different parts with the comma
            parts = line.split(", ")
            description = parts[0]
            timestamp = parts[1]
            score = int(parts[2])
            
            rows.append({
                "description": description,
                "timestamp": timestamp,
                "score": score
            })
            
    return pd.DataFrame(rows)


In [60]:
annotator_1 = parse_file(path_1)
annotator_2 = parse_file(path_2)

annotator_2.head()

,description,timestamp,score
0,A car and four people in the snow with two of ...,00:00 - 00:05,1
1,People and a dog walk through the snow next to...,00:26 - 01:30,0
2,The car is stuck in the snow,01:37 - 01:55,2
3,A kid is trying to push a car but it remains a...,02:12 - 02:33,2
4,Four people and a dog walk back to a different...,02:37 - 03:05,0


In [61]:
def match_descriptions(df1, df2, threshold: float = 0.5):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    #Encode all descriptions
    emb1 = model.encode(df1["description"].tolist(), convert_to_tensor=True)
    emb2 = model.encode(df2["description"].tolist(), convert_to_tensor=True)

    #Cosine similarity matrix
    sim_matrix = util.cos_sim(emb1, emb2)

    matches = []
    
    for index1 in range(len(df1)):
        best_index2  = torch.argmax(sim_matrix[index1]).item()
        best_score = sim_matrix[index1][best_index2].item()

        if best_score >= threshold:
            matches.append({
                "description_1": df1.iloc[index1]["description"],
                "description_2": df2.iloc[best_index2]["description"],
                "similarity": round(best_score, 3),
                "score_1": df1.iloc[index1]["score"],
                "score_2": df2.iloc[best_index2]["score"],
                "timestamp_1": df1.iloc[index1]["timestamp"],
                "timestamp_2": df2.iloc[best_index2]["timestamp"],
            })
            sim_matrix[:, best_index2] = -1  #Put to -1 so it can't be matched again

    return pd.DataFrame(matches)

matched = match_descriptions(annotator_1, annotator_2)

In [62]:
print(matched)

                                       description_1  \
0  People wearing winter clothes and holding shov...   
1  People wearing hoodies and a dog are walking t...   
2  The video shows a car stuck in snow and surrou...   
3  The video shows a dog walking on the snow surr...   
4  The video shows people inside of a car driving...   
5  The video shows people inside of a car driving...   

                                       description_2  similarity  score_1  \
0  A car and four people in the snow with two of ...       0.673        2   
1  Four people and a dog walk back to a different...       0.515        0   
2                       The car is stuck in the snow       0.663        1   
3  People and a dog walk through the snow next to...       0.505       -1   
4              People in a car drive through a field       0.654        2   
5                             A car drives on a road       0.511        2   

   score_2    timestamp_1    timestamp_2  
0        1  00:00 - 00:2

In [63]:
#Consistency in selected events with Cohen's Kappa

n_matched = len(matched)
#Number of events selected by annotator 1/2 only
n_only_a1 = len(annotator_1) - n_matched
n_only_a2 = len(annotator_2) - n_matched

#Makes two binary vector where each position indicates if one event is selected or not
a1 = [1] * n_matched + [1] * n_only_a1 + [0] * n_only_a2
a2 = [1] * n_matched + [0] * n_only_a1 + [1] * n_only_a2

kappa = cohen_kappa_score(a1, a2)
print(f"Cohen's Kappa on event selection: {kappa:.3f}")

Cohen's Kappa on event selection: -0.333


In [64]:
#Temporal boundaries

#Converts "00:44 - 00:58" to (44, 58) in seconds (so, 01:05 is 65)
def parse_timestamp(ts):
    start, end = ts.split(" - ")
    def to_seconds(t):
        m, s = t.strip().split(":")
        return int(m) * 60 + int(s)
    return to_seconds(start), to_seconds(end)

#Computes IoU
def iou(ts1,ts2):
    s1, e1 = parse_timestamp(ts1)
    s2, e2 = parse_timestamp(ts2)
    
    overlap = max(0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)
    
    return overlap / union if union > 0 else 0.0

#Apply to matched pairs
matched["iou"] = matched.apply(
    lambda row: iou(row["timestamp_1"], row["timestamp_2"]), axis=1
)

print(f"Mean IoU on temporal boundaries: {matched['iou'].mean():.3f}")

Mean IoU on temporal boundaries: 0.309


In [65]:
#Quadratic penalizes larger disagreements
kappa_scores = cohen_kappa_score(matched["score_1"], matched["score_2"], weights="quadratic") 
print(f"Cohen's Kappa on subjectivity ratings: {kappa_scores:.3f}")

Cohen's Kappa on subjectivity ratings: 0.462


In [66]:
#Missed events by annotator 1

matched_a2 = set(matched["description_2"])
missed = annotator_2[~annotator_2["description"].isin(matched_a2)]

print(missed["description"].tolist())

['A kid is trying to push a car but it remains at rest', 'Everyone gets into a car', 'A car is started and put into first gear']


In [67]:
#Missed events by annotator 2

matched_r1 = set(matched["description_1"])
missed = annotator_1[~annotator_1["description"].isin(matched_r1)]

print(missed["description"].tolist())

['A woman with black hair and a person wearing a red full-face scarf', 'The video shows a kid wearing a yellow jacket', 'The video shows a person turning a car on using keys']
